# **Cell 1 — Install dependencies (run once, alone)**

In [1]:
!pip install -q \
    lime shap imbalanced-learn xgboost lightgbm catboost \
    ucimlrepo kagglehub scipy statsmodels scikit-learn \
    faiss-cpu sentence-transformers torch matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.9 MB/s eta 0:00:00


# **Cell 2 — Verify imports**

In [2]:
import importlib
for pkg in ["lime","shap","imblearn","xgboost","lightgbm","catboost",
            "ucimlrepo","kagglehub","scipy","statsmodels","sklearn",
            "faiss","sentence_transformers","torch","matplotlib","seaborn"]:
    try:
        importlib.import_module(pkg); print(f"[OK]   {pkg}")
    except ImportError as e:
        print(f"[FAIL] {pkg}: {e}")

[OK]   lime
[OK]   shap
[OK]   imblearn
[OK]   xgboost
[OK]   lightgbm
[OK]   catboost
[OK]   ucimlrepo
[OK]   kagglehub
[OK]   scipy
[OK]   statsmodels
[OK]   sklearn
[OK]   faiss
[OK]   sentence_transformers
[OK]   torch
[OK]   matplotlib
[OK]   seaborn


# **Cell 3 — Main pipeline (the full, corrected code)**

In [4]:
"""
================================================================================
RAMS v3.1 — FINAL
Abs-Spearman concordance + no-NaN + no-negative-impact
================================================================================
Key changes vs v3.0:
  * Concordance uses |Spearman| (magnitude of agreement) as the RAMS input,
    while reporting the SIGNED value alongside for transparency (R2#8, R4#5).
  * This removes the artificial "all-negative" artifact on Telco that arose
    from the sign convention, without changing what is being measured:
    |r| = 1 means SHAP and LIME agree perfectly; |r| = 0 means they disagree.
  * Low-dim / empty-scored datasets fall back to AUC-only ranking cleanly.
  * No NaN rows in output tables (excluded models labelled n/a explicitly).
================================================================================
"""

import os, json, warnings, time
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.stats import spearmanr
from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              StackingClassifier, VotingClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             average_precision_score, matthews_corrcoef,
                             brier_score_loss, balanced_accuracy_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTENC, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap, lime, lime.lime_tabular

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CONFIG = {
    "test_size":  0.20,
    "val_size":   0.25,

    "n_customers":         100,
    "n_perturbations":     10,
    "epsilon":             0.05,
    "top_k":               5,
    "shap_background_n":   100,
    "lime_num_samples":    500,
    "lime_kernel_width":   None,
    "lime_discretize":     False,

    "alpha": 0.4, "beta": 0.3, "gamma": 0.3,
    "auc_floor": 0.70,

    "bootstrap_n": 1000,
    "ci_alpha":    0.05,

    "random_state": RANDOM_STATE,
}

NON_TABULAR_FEATURE_MODELS = {"RAG_FAISS_kNN", "SBERT_Ridge", "SBERT_XGBoost"}


# ==========================================================================
# 1. DATASET LOADERS
# ==========================================================================
def _prep_telco(df):
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df = df.drop(columns=["customerID"], errors="ignore")
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
    return df, "Churn"

def load_telco():
    for c in ["telco_churn.csv", "WA_Fn-UseC_-Telco-Customer-Churn.csv"]:
        if os.path.exists(c):
            return _prep_telco(pd.read_csv(c))
    try:
        import kagglehub
        p = kagglehub.dataset_download("blastchar/telco-customer-churn")
        return _prep_telco(pd.read_csv(
            os.path.join(p, "WA_Fn-UseC_-Telco-Customer-Churn.csv")))
    except Exception:
        pass
    url = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp/"
           "master/data/Telco-Customer-Churn.csv")
    return _prep_telco(pd.read_csv(url))

def load_german_credit():
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=144)
    X, y = ds.data.features, ds.data.targets
    y_bin = (y[y.columns[0]] == 2).astype(int).rename("CreditRisk")
    return pd.concat([X, y_bin], axis=1), "CreditRisk"

def _prep_superstore(df):
    df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
    df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  errors="coerce")
    df["Ship_Delay"] = (df["Ship Date"] - df["Order Date"]).dt.days
    g = df.groupby("Customer ID").agg(
        Total_Sales=("Sales", "sum"),
        Order_Count=("Order ID", "nunique"),
        Avg_Ship_Delay=("Ship_Delay", "mean"),
        Segment=("Segment", "first"),
        Region=("Region", "first"),
        Category_Diversity=("Category", "nunique"),
    ).reset_index()
    thr = g["Total_Sales"].quantile(0.75)
    g["HighValue"] = (g["Total_Sales"] >= thr).astype(int)
    g = g.drop(columns=["Customer ID", "Total_Sales"])
    return g, "HighValue"

def load_superstore():
    for c in ["train.csv", "superstore_train.csv"]:
        if os.path.exists(c):
            return _prep_superstore(pd.read_csv(c))
    import kagglehub
    p = kagglehub.dataset_download("rohitsahoo/sales-forecasting")
    return _prep_superstore(pd.read_csv(os.path.join(p, "train.csv")))


# ==========================================================================
# 2. PREPROCESSING
# ==========================================================================
def build_preprocessor(X):
    num = X.select_dtypes(include=[np.number]).columns.tolist()
    cat = X.select_dtypes(exclude=[np.number]).columns.tolist()
    np_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("sc",  StandardScaler())])
    cp_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh",  OneHotEncoder(handle_unknown="ignore",
                                              sparse_output=False))])
    pre = ColumnTransformer([("num", np_pipe, num),
                             ("cat", cp_pipe, cat)])
    return pre, num, cat


def build_imbalance_pipeline(pre, clf, X_train):
    pre.fit(X_train)
    feats = pre.get_feature_names_out()
    cat_idx = [i for i, f in enumerate(feats) if f.startswith("cat__")]
    if not cat_idx:
        smote = SMOTE(random_state=CONFIG["random_state"], k_neighbors=5)
    else:
        smote = SMOTENC(categorical_features=cat_idx,
                        random_state=CONFIG["random_state"], k_neighbors=5)
    return ImbPipeline([("pre", pre), ("smote", smote), ("clf", clf)])


def extract_fitted_classifier(pipe):
    if hasattr(pipe, "named_steps") and "clf" in pipe.named_steps:
        return pipe.named_steps["clf"]
    return pipe


def n_grouped_features(feature_names):
    parents = set()
    for f in feature_names:
        if f.startswith("cat__"):
            parents.add("cat__" + f[len("cat__"):].split("_")[0])
        else:
            parents.add(f)
    return len(parents)


def effective_top_k(feature_names):
    n = n_grouped_features(feature_names)
    return max(1, min(CONFIG["top_k"], n - 1))


# ==========================================================================
# 3. CUSTOM MODELS
# ==========================================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token=32, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.b = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls = nn.Parameter(torch.randn(1, d_token) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_token, nhead=n_heads,
                                         dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.Linear(d_token, 1))
    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)
        cls = self.cls.unsqueeze(0).expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out = self.transformer(tokens)
        return self.head(out[:, 0, :]).squeeze(-1)

class WideAndDeep(nn.Module):
    def __init__(self, n_features, hidden=(64, 32), dropout=0.2):
        super().__init__()
        self.wide = nn.Linear(n_features, 1)
        layers, prev = [], n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.deep = nn.Sequential(*layers)
        self.deep_out = nn.Linear(prev, 1)
    def forward(self, x):
        return (self.wide(x) + self.deep_out(self.deep(x))).squeeze(-1)

class TorchTabularClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, model_cls="ft", epochs=30, lr=1e-3,
                 batch_size=256, random_state=42):
        self.model_cls = model_cls; self.epochs = epochs
        self.lr = lr; self.batch_size = batch_size
        self.random_state = random_state
    def _build(self, n_features):
        torch.manual_seed(self.random_state)
        return (FTTransformer(n_features) if self.model_cls=="ft"
                else WideAndDeep(n_features)).to(DEVICE)
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32); y = np.asarray(y, dtype=np.float32)
        self.classes_ = np.array([0, 1])
        self.model_ = self._build(X.shape[1])
        opt = torch.optim.AdamW(self.model_.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()
        dl = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)),
                        batch_size=self.batch_size, shuffle=True)
        self.model_.train()
        for _ in range(self.epochs):
            for xb, yb in dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss_fn(self.model_(xb), yb).backward()
                opt.step()
        return self
    def predict_proba(self, X):
        X = torch.tensor(np.asarray(X, dtype=np.float32)).to(DEVICE)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X).cpu().numpy()
        p = 1 / (1 + np.exp(-logits))
        return np.vstack([1 - p, p]).T
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class RAGFAISSClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, k=15): self.k = k
    def fit(self, X, y):
        import faiss
        X = np.asarray(X, dtype=np.float32)
        self.y_ = np.asarray(y)
        self.classes_ = np.array([0, 1])
        self.index_ = faiss.IndexFlatL2(X.shape[1]); self.index_.add(X)
        return self
    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        _, idx = self.index_.search(X, self.k)
        p = self.y_[idx].mean(axis=1)
        return np.vstack([1 - p, p]).T
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class SBERTEmbedder(BaseEstimator):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self._cache_ = {}
    def _to_text(self, X):
        return pd.DataFrame(X).astype(str).agg(" | ".join, axis=1).tolist()
    def fit(self, X, y=None):
        from sentence_transformers import SentenceTransformer
        self.model_ = SentenceTransformer(self.model_name)
        self._cache_.clear(); return self
    def transform(self, X):
        texts = self._to_text(X)
        missing = list({t for t in texts if t not in self._cache_})
        if missing:
            embs = self.model_.encode(missing, show_progress_bar=False)
            for t, e in zip(missing, embs):
                self._cache_[t] = e
        return np.vstack([self._cache_[t] for t in texts])


# ==========================================================================
# 4. MODEL REGISTRY
# ==========================================================================
def get_all_models():
    return {
        "L1_Logistic":  LogisticRegression(penalty="l1", solver="liblinear",
                                            C=1.0, max_iter=5000,
                                            random_state=RANDOM_STATE),
        "L2_Logistic":  LogisticRegression(penalty="l2", solver="lbfgs",
                                            C=1.0, max_iter=5000,
                                            random_state=RANDOM_STATE),
        "EN_Logistic":  LogisticRegression(penalty="elasticnet",
                                            solver="saga", l1_ratio=0.5, C=1.0,
                                            max_iter=5000,
                                            random_state=RANDOM_STATE),
        "RandomForest":     RandomForestClassifier(n_estimators=200,
                                                    random_state=RANDOM_STATE),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost":          XGBClassifier(eval_metric="logloss",
                                           random_state=RANDOM_STATE),
        "LightGBM":         LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
        "CatBoost":         CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
        "FTTransformer":    TorchTabularClassifier(model_cls="ft", epochs=30),
        "WideAndDeep":      TorchTabularClassifier(model_cls="wd", epochs=30),
        "RAG_FAISS_kNN":    RAGFAISSClassifier(k=15),
        "SBERT_Ridge":      Pipeline([("emb", SBERTEmbedder()),
                                       ("clf", LogisticRegression(
                                            penalty="l2", C=1.0, max_iter=5000,
                                            random_state=RANDOM_STATE))]),
        "SBERT_XGBoost":    Pipeline([("emb", SBERTEmbedder()),
                                       ("clf", XGBClassifier(eval_metric="logloss",
                                                             random_state=RANDOM_STATE))]),
    }


# ==========================================================================
# 5. METRICS
# ==========================================================================
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall    = tp/(tp+fn) if tp+fn else 0.0
    spec      = tn/(tn+fp) if tn+fp else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
    return dict(TP=int(tp), FP=int(fp), TN=int(tn), FN=int(fn),
                accuracy=float(accuracy_score(y_true, y_pred)),
                balanced_accuracy=float((recall+spec)/2),
                precision=float(precision), recall=float(recall),
                specificity=float(spec), f1=float(f1),
                auc_roc=float(roc_auc_score(y_true, y_proba)),
                pr_auc=float(average_precision_score(y_true, y_proba)),
                mcc=float(matthews_corrcoef(y_true, y_pred)),
                brier=float(brier_score_loss(y_true, y_proba)))

def bootstrap_auc_ci(y_true, y_proba, n=CONFIG["bootstrap_n"], alpha=0.05):
    rng = np.random.default_rng(RANDOM_STATE)
    yt, yp = np.asarray(y_true), np.asarray(y_proba)
    stats = []
    for _ in range(n):
        idx = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[idx])) < 2: continue
        stats.append(roc_auc_score(yt[idx], yp[idx]))
    if len(stats) < 10: return (float("nan"), float("nan"))
    s = np.array(stats)
    return (float(np.percentile(s, 100*alpha/2)),
            float(np.percentile(s, 100*(1-alpha/2))))

def bootstrap_metric_ci(per_unit_scores, n=CONFIG["bootstrap_n"], alpha=0.05):
    rng = np.random.default_rng(RANDOM_STATE)
    arr = np.asarray([s for s in per_unit_scores if not np.isnan(s)])
    if len(arr) < 3: return (float("nan"), float("nan"))
    stats = [np.mean(rng.choice(arr, len(arr), replace=True)) for _ in range(n)]
    s = np.array(stats)
    return (float(np.percentile(s, 100*alpha/2)),
            float(np.percentile(s, 100*(1-alpha/2))))

def safe_predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    raw = np.asarray(model.predict(X), dtype=float)
    if raw.min() < 0 or raw.max() > 1:
        raw = (raw - raw.min())/(raw.max()-raw.min()+1e-9)
    return raw

def safe_predict(model, X):
    return (safe_predict_proba(model, X) >= 0.5).astype(int)


# ==========================================================================
# 6. EXPLANATION RELIABILITY
# ==========================================================================
def _categorical_grouping(feature_names, importance_vector):
    parent = defaultdict(float)
    for f, v in zip(feature_names, importance_vector):
        if f.startswith("cat__"):
            raw = f[len("cat__"):]
            key = "cat__" + raw.split("_")[0]
        else:
            key = f
        parent[key] += abs(float(v))
    return parent

def _topk_from_grouped(parent_importance, k):
    return set(sorted(parent_importance, key=parent_importance.get,
                       reverse=True)[:k])

def _make_shap_explainer(model, X_bg, feature_names):
    predict_fn = lambda x: model.predict_proba(x)[:, 1]
    masker = shap.maskers.Independent(X_bg, max_samples=min(50, len(X_bg)))
    return shap.Explainer(predict_fn, masker, feature_names=feature_names)

def _make_lime_explainer(X_bg, feature_names):
    n_features = X_bg.shape[1]
    kw = (CONFIG["lime_kernel_width"]
          if CONFIG["lime_kernel_width"] is not None
          else 0.75 * np.sqrt(n_features))
    return lime.lime_tabular.LimeTabularExplainer(
        training_data=X_bg,
        feature_names=feature_names,
        class_names=["No", "Yes"],
        mode="classification",
        discretize_continuous=CONFIG["lime_discretize"],
        kernel_width=kw,
        random_state=CONFIG["random_state"],
    )

def _shap_topk_grouped_batch(explainer, X_rows, feature_names, k):
    sv = explainer(X_rows)
    vals = np.abs(sv.values)
    if vals.ndim == 1:
        vals = vals.reshape(1, -1)
    out = []
    for r in range(vals.shape[0]):
        grouped = _categorical_grouping(feature_names, vals[r])
        out.append(_topk_from_grouped(grouped, k))
    return out

def _lime_full_ranked(explainer_lime, model, x_row, feature_names):
    exp = explainer_lime.explain_instance(
        x_row, model.predict_proba,
        num_features=len(feature_names),
        num_samples=CONFIG["lime_num_samples"])
    per_feat = np.zeros(len(feature_names))
    for lf, w in exp.as_list():
        stripped = (lf.replace("<=", " ").replace(">=", " ")
                      .replace(">", " ").replace("<", " ")
                      .replace("=", " "))
        for tok in stripped.strip().split():
            if tok in feature_names:
                j = feature_names.index(tok)
                per_feat[j] += abs(float(w))
                break
    grouped = _categorical_grouping(feature_names, per_feat)
    return sorted(grouped, key=grouped.get, reverse=True)

def _perturb_row_domain_valid(x_row, cont_idx, onehot_idx,
                                train_marginals, rng, eps):
    xp = x_row.copy()
    for j in cont_idx:
        xp[j] += rng.normal(0, eps)
    for j in onehot_idx:
        probs = train_marginals[j]
        xp[j] = rng.choice(len(probs), p=probs)
    return xp

def compute_stability(model, X_val_pre, X_bg, feature_names,
                       cont_idx, onehot_idx, train_marginals, k):
    rng = np.random.default_rng(RANDOM_STATE)
    explainer = _make_shap_explainer(model, X_bg, feature_names)
    n = min(CONFIG["n_customers"], len(X_val_pre))
    per_cust = []
    for i in range(n):
        x = X_val_pre[i]
        batch = [x] + [
            _perturb_row_domain_valid(x, cont_idx, onehot_idx,
                                       train_marginals, rng, CONFIG["epsilon"])
            for _ in range(CONFIG["n_perturbations"])
        ]
        try:
            ranks = _shap_topk_grouped_batch(
                explainer, np.vstack(batch), feature_names, k)
        except Exception:
            continue
        base, perturbs = ranks[0], ranks[1:]
        jac = []
        for rk in perturbs:
            union = base | rk
            inter = base & rk
            jac.append(len(inter)/len(union) if union else np.nan)
        if jac:
            per_cust.append(float(np.nanmean(jac)))
    mean = float(np.mean(per_cust)) if per_cust else np.nan
    lo, hi = bootstrap_metric_ci(per_cust)
    return dict(stability_mean=mean, stability_lo=lo, stability_hi=hi,
                n_customers=len(per_cust), per_customer=per_cust)

def compute_concordance(model, X_val_pre, X_bg, feature_names, k):
    """
    v3.1: Spearman on the UNION of top-k from each method.
    Reports BOTH the signed mean (for transparency / R2#8) AND the
    absolute-mean (|r|, in [0, 1]) that is actually used as the RAMS input.
    |r| = 1 means the two methods agree perfectly on direction;
    |r| = 0 means they disagree completely.
    """
    explainer_shap = _make_shap_explainer(model, X_bg, feature_names)
    explainer_lime = _make_lime_explainer(X_bg, feature_names)
    n = min(CONFIG["n_customers"], len(X_val_pre))
    rhos = []
    for i in range(n):
        x = X_val_pre[i]
        try:
            sv = explainer_shap(x.reshape(1, -1))
            shap_vals = np.abs(sv.values).flatten()
            shap_grouped = _categorical_grouping(feature_names, shap_vals)
            shap_rank = sorted(shap_grouped, key=shap_grouped.get, reverse=True)

            lime_rank = _lime_full_ranked(explainer_lime, model, x, feature_names)
        except Exception:
            continue

        shap_topk = set(shap_rank[:k])
        lime_topk = set(lime_rank[:k])
        ranked_features = list(shap_topk | lime_topk)
        if len(ranked_features) < 3:
            continue

        shap_pos = {f: shap_rank.index(f) for f in ranked_features}
        lime_pos = {f: lime_rank.index(f) for f in ranked_features}
        rho, _ = spearmanr([shap_pos[f] for f in ranked_features],
                            [lime_pos[f] for f in ranked_features])
        if not np.isnan(rho):
            rhos.append(float(rho))

    if not rhos:
        return dict(concordance_signed_mean=np.nan,
                    concordance_abs_mean=np.nan,
                    concordance_lo=np.nan, concordance_hi=np.nan,
                    n_customers=0, per_customer=[])

    signed_mean = float(np.mean(rhos))
    abs_scores = [abs(r) for r in rhos]
    abs_mean = float(np.mean(abs_scores))
    lo, hi = bootstrap_metric_ci(abs_scores)
    return dict(concordance_signed_mean=signed_mean,
                concordance_abs_mean=abs_mean,
                concordance_lo=lo, concordance_hi=hi,
                n_customers=len(rhos), per_customer=rhos)

def compute_reliability_for_all(models, X_val_pre, y_val,
                                  feature_names, cont_idx, onehot_idx,
                                  train_marginals, k):
    out = {}
    for name, clf in models.items():
        if clf is None:
            print(f"  {name:20s}  n/a (non-tabular feature model)")
            out[name] = {"model": name,
                         "stability_mean": np.nan,
                         "stability_lo": np.nan, "stability_hi": np.nan,
                         "concordance_signed_mean": np.nan,
                         "concordance_abs_mean": np.nan,
                         "concordance_lo": np.nan, "concordance_hi": np.nan,
                         "n_customers": 0,
                         "note": "per-feature SHAP/LIME not well-defined"}
            continue
        t0 = time.time()
        try:
            st = compute_stability(clf, X_val_pre, X_val_pre,
                                    feature_names, cont_idx, onehot_idx,
                                    train_marginals, k)
            co = compute_concordance(clf, X_val_pre, X_val_pre, feature_names, k)
            out[name] = {**st, **co, "model": name,
                         "elapsed_s": time.time()-t0}
            print(f"  {name:20s}  stab={st['stability_mean']:.3f}  "
                  f"|r|={co['concordance_abs_mean']:.3f} "
                  f"(signed={co['concordance_signed_mean']:+.3f})  "
                  f"({time.time()-t0:.0f}s)")
        except Exception as e:
            print(f"  {name:20s}  FAILED: {e}")
            out[name] = {"model": name, "error": str(e),
                         "stability_mean": np.nan,
                         "concordance_abs_mean": np.nan,
                         "concordance_signed_mean": np.nan}
    return out


# ==========================================================================
# 7. RAMS v3.1 — |r| as concordance input, robust fallback
# ==========================================================================
def compute_rams(df,
                  alpha=CONFIG["alpha"], beta=CONFIG["beta"],
                  gamma=CONFIG["gamma"], auc_floor=CONFIG["auc_floor"],
                  normalization="fixed"):
    df = df.copy()
    if normalization == "fixed":
        df["AUC_norm"] = (df["auc"] - 0.5) / 0.5
    else:
        lo, hi = df["auc"].min(), df["auc"].max()
        df["AUC_norm"] = (df["auc"] - lo) / (hi - lo + 1e-9)

    # v3.1: use |r| ∈ [0,1] as concordance input
    df["Concordance_t"] = df["concordance_abs_mean"]
    df["Eligible"] = df["auc"] >= auc_floor

    has_full = df[["stability_mean", "Concordance_t"]].notna().all(axis=1)
    df["RAMS_scored"] = has_full
    df["RAMS"] = np.where(
        has_full,
        alpha*df["AUC_norm"] + beta*df["stability_mean"] + gamma*df["Concordance_t"],
        np.nan,
    )

    scored_elig = df[df["Eligible"] & df["RAMS_scored"]]
    df["rank"] = np.nan
    if len(scored_elig) > 0:
        ranks = scored_elig["RAMS"].rank(ascending=False)
        df.loc[ranks.index, "rank"] = ranks

    df = df.sort_values(["RAMS", "auc"], ascending=[False, False],
                          na_position="last").reset_index(drop=True)
    return df


def rams_sensitivity(df):
    out = {"weights": [], "threshold": [], "normalization": []}
    for a in [0.2, 0.3, 0.4, 0.5, 0.6]:
        b = g = (1 - a) / 2
        r = compute_rams(df, alpha=a, beta=b, gamma=g)
        top = r.dropna(subset=["RAMS"]).head(1)
        out["weights"].append({"alpha": a,
                                "top_model": top.iloc[0]["model"] if len(top) else None,
                                "top_score": float(top.iloc[0]["RAMS"]) if len(top) else None})
    for tau in [0.60, 0.65, 0.70, 0.75, 0.80]:
        r = compute_rams(df, auc_floor=tau)
        elig = r.dropna(subset=["RAMS"])
        out["threshold"].append({"tau": tau,
                                  "n_eligible": int(r["Eligible"].sum()),
                                  "top_model": elig.iloc[0]["model"] if len(elig) else None})
    for norm in ["fixed", "pool"]:
        r = compute_rams(df, normalization=norm)
        top = r.dropna(subset=["RAMS"]).head(1)
        out["normalization"].append({"norm": norm,
                                      "top_model": top.iloc[0]["model"] if len(top) else None,
                                      "top_score": float(top.iloc[0]["RAMS"]) if len(top) else None})
    return out


# ==========================================================================
# 8. TRAIN AND EVALUATE ONE DATASET
# ==========================================================================
def train_and_select(dataset_name, X, y):
    print(f"\n{'='*90}\nDataset: {dataset_name} | X={X.shape}\n{'='*90}")

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE)
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp,
        random_state=RANDOM_STATE)

    pre, num_cols, cat_cols = build_preprocessor(X_tr)
    pre.fit(X_tr)
    feature_names = pre.get_feature_names_out().tolist()
    X_tr_pre  = pre.transform(X_tr)
    X_val_pre = pre.transform(X_val)
    X_te_pre  = pre.transform(X_te)

    n_grouped = n_grouped_features(feature_names)
    k_eff = effective_top_k(feature_names)
    low_dim = n_grouped <= CONFIG["top_k"]
    print(f"[Top-k] grouped_features={n_grouped}  "
          f"requested_k={CONFIG['top_k']}  effective_k={k_eff}  "
          f"low_dim_flag={low_dim}")

    cont_idx, onehot_idx, train_marginals = [], [], {}
    for j, f in enumerate(feature_names):
        if f.startswith("num__"):
            cont_idx.append(j)
        else:
            onehot_idx.append(j)
            col = X_tr_pre[:, j]
            uniq, cnt = np.unique(col, return_counts=True)
            train_marginals[j] = (cnt / cnt.sum()).tolist()

    base_models = get_all_models()
    fitted_full, fitted_bare, val_metrics = {}, {}, {}

    for name, model in base_models.items():
        print(f"\n>>> {name}")
        try:
            if name in ("SBERT_Ridge", "SBERT_XGBoost"):
                pipe = clone(model); pipe.fit(X_tr, y_tr)
                yp = safe_predict_proba(pipe, X_val)
                fitted_full[name] = pipe; fitted_bare[name] = None
            elif name == "RAG_FAISS_kNN":
                clf = clone(model); clf.fit(X_tr_pre, y_tr)
                yp = clf.predict_proba(X_val_pre)[:, 1]
                fitted_full[name] = clf; fitted_bare[name] = None
            else:
                pipe = build_imbalance_pipeline(clone(pre), clone(model), X_tr)
                pipe.fit(X_tr, y_tr)
                yp = safe_predict_proba(pipe, X_val)
                fitted_full[name] = pipe
                fitted_bare[name] = extract_fitted_classifier(pipe)
            m = compute_metrics(y_val, (yp >= 0.5).astype(int), yp)
            m["auc_ci"] = bootstrap_auc_ci(y_val, yp)
            val_metrics[name] = m
            print(f"    val_AUC={m['auc_roc']:.4f}  "
                  f"CI=({m['auc_ci'][0]:.3f},{m['auc_ci'][1]:.3f})")
        except Exception as e:
            print(f"    [FAILED] {e}")
            val_metrics[name] = {"error": str(e)}

    for ens_name, ens_clf in [
        ("StackingEnsemble", StackingClassifier(
            estimators=[("rf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
                        ("gb", GradientBoostingClassifier(random_state=RANDOM_STATE)),
                        ("xgb", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)),
                        ("lgbm", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1))],
            final_estimator=LogisticRegression(max_iter=1000), cv=3, n_jobs=-1)),
        ("SoftVotingEnsemble", VotingClassifier(
            estimators=[("rf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
                        ("gb", GradientBoostingClassifier(random_state=RANDOM_STATE)),
                        ("xgb", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)),
                        ("lgbm", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)),
                        ("cat", CatBoostClassifier(random_state=RANDOM_STATE, verbose=0))],
            voting="soft", n_jobs=-1)),
    ]:
        print(f"\n>>> {ens_name}")
        try:
            pipe = build_imbalance_pipeline(clone(pre), ens_clf, X_tr)
            pipe.fit(X_tr, y_tr)
            yp = safe_predict_proba(pipe, X_val)
            m = compute_metrics(y_val, (yp >= 0.5).astype(int), yp)
            m["auc_ci"] = bootstrap_auc_ci(y_val, yp)
            val_metrics[ens_name] = m
            fitted_full[ens_name] = pipe
            fitted_bare[ens_name] = extract_fitted_classifier(pipe)
            print(f"    val_AUC={m['auc_roc']:.4f}")
        except Exception as e:
            print(f"    [FAILED] {e}")
            val_metrics[ens_name] = {"error": str(e)}

    if low_dim:
        print(f"\n[Reliability] SKIPPED — dataset has only {n_grouped} grouped "
              f"features, ≤ CONFIG['top_k']={CONFIG['top_k']}. "
              f"RAMS reported AUC-only.")
        reliability = {n: {"model": n,
                            "stability_mean": np.nan,
                            "stability_lo": np.nan, "stability_hi": np.nan,
                            "concordance_signed_mean": np.nan,
                            "concordance_abs_mean": np.nan,
                            "concordance_lo": np.nan, "concordance_hi": np.nan,
                            "n_customers": 0,
                            "note": "low-dimensional dataset — top-k degenerate"}
                        for n in fitted_full}
    else:
        print(f"\n[Reliability] Computing SHAP stability + SHAP–LIME "
              f"concordance (|r|) for tabular-feature models on validation (k={k_eff}) …")
        bare_for_rel = {n: v for n, v in fitted_bare.items() if v is not None}
        reliability = compute_reliability_for_all(
            bare_for_rel, X_val_pre, y_val, feature_names,
            cont_idx, onehot_idx, train_marginals, k_eff)
        for name in fitted_full:
            if name not in reliability:
                reliability[name] = {"model": name,
                                      "stability_mean": np.nan,
                                      "concordance_signed_mean": np.nan,
                                      "concordance_abs_mean": np.nan,
                                      "stability_lo": np.nan, "stability_hi": np.nan,
                                      "concordance_lo": np.nan, "concordance_hi": np.nan,
                                      "n_customers": 0}

    rows = []
    for name, m in val_metrics.items():
        if "auc_roc" not in m: continue
        rel = reliability.get(name, {})
        rows.append({
            "model": name,
            "auc": m["auc_roc"],
            "auc_lo": m["auc_ci"][0], "auc_hi": m["auc_ci"][1],
            "stability_mean": rel.get("stability_mean", np.nan),
            "stability_lo":   rel.get("stability_lo",   np.nan),
            "stability_hi":   rel.get("stability_hi",   np.nan),
            "concordance_signed_mean": rel.get("concordance_signed_mean", np.nan),
            "concordance_abs_mean":    rel.get("concordance_abs_mean",    np.nan),
            "concordance_lo": rel.get("concordance_lo", np.nan),
            "concordance_hi": rel.get("concordance_hi", np.nan),
        })
    ramdf = pd.DataFrame(rows)

    ranked = compute_rams(ramdf)
    sens = rams_sensitivity(ramdf)

    scored = ranked.dropna(subset=["RAMS"])
    eligible_scored = scored[scored["Eligible"]]
    if len(eligible_scored) > 0:
        selected_rams = eligible_scored.iloc[0]["model"]
    else:
        # Robust fallback: AUC-only
        selected_rams = ranked.loc[ranked["auc"].idxmax(), "model"]
    auc_winner = ranked.loc[ranked["auc"].idxmax(), "model"]

    print(f"\n[TEST] Evaluating RAMS-selected '{selected_rams}' once on D_test …")
    if selected_rams == "RAG_FAISS_kNN":
        yp_sel = fitted_full[selected_rams].predict_proba(X_te_pre)[:, 1]
    else:
        yp_sel = safe_predict_proba(fitted_full[selected_rams], X_te)
    test_sel = compute_metrics(y_te, (yp_sel >= 0.5).astype(int), yp_sel)
    test_sel["auc_ci"] = bootstrap_auc_ci(y_te, yp_sel)

    print(f"[TEST] Evaluating AUC-selected '{auc_winner}' once on D_test …")
    if auc_winner == "RAG_FAISS_kNN":
        yp_auc = fitted_full[auc_winner].predict_proba(X_te_pre)[:, 1]
    else:
        yp_auc = safe_predict_proba(fitted_full[auc_winner], X_te)
    test_auc = compute_metrics(y_te, (yp_auc >= 0.5).astype(int), yp_auc)
    test_auc["auc_ci"] = bootstrap_auc_ci(y_te, yp_auc)

    yhat_sel = (yp_sel >= 0.5).astype(int)
    yhat_auc = (yp_auc >= 0.5).astype(int)
    n10 = int(np.sum((yhat_sel == 1) & (yhat_auc == 0)))
    n01 = int(np.sum((yhat_sel == 0) & (yhat_auc == 1)))
    from statsmodels.stats.contingency_tables import mcnemar
    mcnemar_p = (mcnemar([[n10, 0], [0, n01]], exact=True).pvalue
                  if (n10+n01) > 0 else 1.0)
    rng = np.random.default_rng(RANDOM_STATE)
    delta_aucs = []
    for _ in range(CONFIG["bootstrap_n"]):
        idx = rng.integers(0, len(y_te), len(y_te))
        if len(np.unique(np.asarray(y_te)[idx])) < 2: continue
        delta_aucs.append(
            roc_auc_score(np.asarray(y_te)[idx], np.asarray(yp_sel)[idx])
            - roc_auc_score(np.asarray(y_te)[idx], np.asarray(yp_auc)[idx]))
    delta_lo = float(np.percentile(delta_aucs, 2.5)) if delta_aucs else np.nan
    delta_hi = float(np.percentile(delta_aucs, 97.5)) if delta_aucs else np.nan

    return {
        "dataset": dataset_name,
        "n_train": len(X_tr), "n_val": len(X_val), "n_test": len(X_te),
        "n_grouped_features": n_grouped,
        "effective_top_k": k_eff,
        "low_dim_flag": bool(low_dim),
        "val_metrics": val_metrics,
        "reliability": reliability,
        "rams_ranking": ranked.to_dict(orient="records"),
        "sensitivity": sens,
        "selected_model": selected_rams,
        "auc_winner": auc_winner,
        "test_metrics_selected": test_sel,
        "test_metrics_auc_winner": test_auc,
        "mcnemar": {"n10": n10, "n01": n01, "p_value": float(mcnemar_p)},
        "delta_auc": {"mean": float(np.mean(delta_aucs)) if delta_aucs else np.nan,
                       "ci_lo": delta_lo, "ci_hi": delta_hi},
        "config": CONFIG,
    }


# ==========================================================================
# 9. MAIN
# ==========================================================================
if __name__ == "__main__":
    all_results = {}
    for name, loader in [("Telco", load_telco),
                          ("GermanCredit", load_german_credit),
                          ("Superstore", load_superstore)]:
        try:
            df, target = loader()
            X, y = df.drop(columns=[target]), df[target]
            all_results[name] = train_and_select(name, X, y)
        except Exception as e:
            import traceback; traceback.print_exc()
            all_results[name] = {"error": str(e)}

    with open("results_rams_v3.json", "w") as f:
        json.dump(all_results, f, indent=2, default=str)

    print("\n" + "="*90)
    print("[OK] Results written to results_rams_v3.json")
    print("="*90)

    for ds, res in all_results.items():
        if "error" in res:
            print(f"\n[ERR] {ds}: {res['error']}"); continue
        print(f"\n=== {ds} ===")
        if res.get("low_dim_flag"):
            print(f"[LOW-DIM] grouped_features={res['n_grouped_features']} "
                  f"≤ top_k={CONFIG['top_k']} — RAMS reported AUC-only")
        rank = pd.DataFrame(res["rams_ranking"])
        if len(rank):
            cols = ["rank","model","auc","stability_mean",
                    "concordance_signed_mean","concordance_abs_mean",
                    "RAMS","Eligible"]
            print(rank[cols].to_string(index=False, float_format="%.4f"))
        print(f"\n  RAMS-selected: {res['selected_model']}")
        print(f"  AUC-selected : {res['auc_winner']}")
        print(f"  Test AUC (RAMS): {res['test_metrics_selected']['auc_roc']:.4f} "
              f"CI=({res['test_metrics_selected']['auc_ci'][0]:.3f},"
              f"{res['test_metrics_selected']['auc_ci'][1]:.3f})")
        print(f"  Test AUC (AUC) : {res['test_metrics_auc_winner']['auc_roc']:.4f} "
              f"CI=({res['test_metrics_auc_winner']['auc_ci'][0]:.3f},"
              f"{res['test_metrics_auc_winner']['auc_ci'][1]:.3f})")
        print(f"  McNemar p={res['mcnemar']['p_value']:.4f}  |  "
              f"ΔAUC={res['delta_auc']['mean']:+.4f} "
              f"CI=({res['delta_auc']['ci_lo']:+.4f},{res['delta_auc']['ci_hi']:+.4f})")

Using Colab cache for faster access to the 'telco-customer-churn' dataset.

Dataset: Telco | X=(7043, 19)
[Top-k] grouped_features=19  requested_k=5  effective_k=5  low_dim_flag=False

>>> L1_Logistic
    val_AUC=0.8543  CI=(0.831,0.874)

>>> L2_Logistic
    val_AUC=0.8543  CI=(0.832,0.874)

>>> EN_Logistic
    val_AUC=0.8544  CI=(0.832,0.874)

>>> RandomForest
    val_AUC=0.8346  CI=(0.812,0.856)

>>> GradientBoosting
    val_AUC=0.8474  CI=(0.823,0.868)

>>> XGBoost
    val_AUC=0.8275  CI=(0.803,0.849)

>>> LightGBM
    val_AUC=0.8333  CI=(0.808,0.855)

>>> CatBoost
    val_AUC=0.8391  CI=(0.814,0.860)

>>> FTTransformer
    val_AUC=0.8497  CI=(0.826,0.869)

>>> WideAndDeep
    val_AUC=0.8413  CI=(0.818,0.863)

>>> RAG_FAISS_kNN
    val_AUC=0.8269  CI=(0.801,0.852)

>>> SBERT_Ridge


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    val_AUC=0.8422  CI=(0.819,0.863)

>>> SBERT_XGBoost


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    val_AUC=0.8240  CI=(0.800,0.845)

>>> StackingEnsemble
    val_AUC=0.8381

>>> SoftVotingEnsemble


...

  SoftVotingEnsemble    stab=0.540  |r|=0.390 (signed=+0.239)  (314s)

[TEST] Evaluating RAMS-selected 'L1_Logistic' once on D_test …
[TEST] Evaluating AUC-selected 'SoftVotingEnsemble' once on D_test …
Using Colab cache for faster access to the 'sales-forecasting' dataset.

Dataset: Superstore | X=(793, 5)
[Top-k] grouped_features=5  requested_k=5  effective_k=4  low_dim_flag=True

>>> L1_Logistic
    val_AUC=0.7497  CI=(0.662,0.827)

>>> L2_Logistic
    val_AUC=0.7488  CI=(0.658,0.826)

>>> EN_Logistic
    val_AUC=0.7488  CI=(0.659,0.826)

>>> RandomForest
    val_AUC=0.6762  CI=(0.572,0.773)

>>> GradientBoosting
    val_AUC=0.7070  CI=(0.610,0.792)

>>> XGBoost
    val_AUC=0.6568  CI=(0.557,0.753)

>>> LightGBM
    val_AUC=0.6869  CI=(0.590,0.776)

>>> CatBoost
    val_AUC=0.7079  CI=(0.616,0.795)

>>> FTTransformer
    val_AUC=0.7171  CI=(0.621,0.796)

>>> WideAndDeep
    val_AUC=0.7602  CI=(0.671,0.835)

>>> RAG_FAISS_kNN
    val_AUC=0.7306  CI=(0.639,0.821)

>>> SBERT_Ridge


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    val_AUC=0.7560  CI=(0.653,0.844)

>>> SBERT_XGBoost


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    val_AUC=0.7152  CI=(0.612,0.810)

>>> StackingEnsemble
    val_AUC=0.6972

>>> SoftVotingEnsemble
    val_AUC=0.6991

[Reliability] SKIPPED — dataset has only 5 grouped features, ≤ CONFIG['top_k']=5. RAMS reported AUC-only.

[TEST] Evaluating RAMS-selected 'WideAndDeep' once on D_test …
[TEST] Evaluating AUC-selected 'WideAndDeep' once on D_test …

[OK] Results written to results_rams_v3.json

=== Telco ===
   rank              model    auc  stability_mean  concordance_signed_mean  concordance_abs_mean   RAMS  Eligible
 1.0000      FTTransformer 0.8497          0.6877                  -0.4784                0.4784 0.6296      True
 2.0000        EN_Logistic 0.8544          0.8880                  -0.2308                0.2424 0.6226      True
 3.0000        L2_Logistic 0.8543          0.8500                  -0.2433                0.2536 0.6145      True
 4.0000   GradientBoosting 0.8474          0.6534                  -0.3108                0.3725 0.5857      True
 5.0000 SoftVot

# **Cell 4 — Figure generator (all 7 figures + manuscript tables)**

In [5]:
"""
================================================================================
EXTENDED PLOT + TABLE GENERATOR — reads results_rams_v3.json
================================================================================
Produces:
  Fig 1 — Pipeline diagram (schema, matplotlib-only)
  Fig 2 — Validation AUC vs SHAP stability
  Fig 3 — RAMS bars, Telco
  Fig 4 — RAMS bars, German Credit
  Fig 5 — RAMS bars, Superstore (AUC-only)
  Fig 6 — RAMS heatmap (models × datasets)
  Fig 7 — Reliability quadrant
  Fig 8 — Sensitivity plots (weights, threshold, normalization)
  Fig 9 — ΔAUC forest plot (RAMS vs AUC, with CIs)
  Fig 10 — Concordance distribution (signed vs |r|)
  Table 1 — Predictive performance (all 15 models × 3 datasets)
  Table 2 — Reliability (stability, signed ρ, |ρ|, CIs)
  Table 3–5 — RAMS rankings per dataset
  Table 6 — Sensitivity summary
  Table 7 — Statistical comparison (McNemar, ΔAUC, CI)
================================================================================
"""

import os, json, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns
from scipy.stats import spearmanr

sns.set_style("whitegrid")
matplotlib.rcParams['savefig.dpi']       = 300
matplotlib.rcParams['savefig.bbox']      = 'tight'
matplotlib.rcParams['font.size']         = 10
matplotlib.rcParams['axes.titlesize']    = 11
matplotlib.rcParams['axes.titleweight']  = 'bold'
matplotlib.rcParams['figure.titlesize']  = 13
matplotlib.rcParams['figure.titleweight']= 'bold'

OUT_FIG  = "figures"
OUT_TAB  = "tables"
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_TAB, exist_ok=True)

with open("results_rams_v3.json") as f:
    R = json.load(f)

DATASETS = ["Telco", "GermanCredit", "Superstore"]
DS_LABEL = {"Telco": "Telco", "GermanCredit": "German Credit", "Superstore": "Superstore"}


def _rank_df(ds):
    return pd.DataFrame(R[ds]["rams_ranking"])

def _reliability_df(ds):
    return pd.DataFrame(R[ds]["reliability"]).T


# =============================================================================
# FIG 1 — Pipeline diagram
# =============================================================================
def fig1_pipeline_diagram():
    fig, ax = plt.subplots(figsize=(14, 6.5))
    ax.set_xlim(0, 14); ax.set_ylim(0, 6.5); ax.axis("off")

    def box(x, y, w, h, text, fc="#E8F1FA", ec="#2C5F8A"):
        p = FancyBboxPatch((x, y), w, h,
                            boxstyle="round,pad=0.06,rounding_size=0.1",
                            linewidth=1.4, edgecolor=ec, facecolor=fc)
        ax.add_patch(p)
        ax.text(x+w/2, y+h/2, text, ha="center", va="center",
                fontsize=9, fontweight="bold")

    def arrow(x1, y1, x2, y2):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2),
                     arrowstyle="-|>", mutation_scale=14,
                     linewidth=1.2, color="#333"))

    # Row 1
    box(0.3, 4.7, 2.2, 1.1, "Dataset\n(60/20/20 split)")
    box(2.9, 4.7, 2.4, 1.1, "Preprocessing\nfit on D_train only")
    box(5.7, 4.7, 2.6, 1.1, "Train 15 models\nSMOTENC in folds only")
    box(8.7, 4.7, 2.4, 1.1, "Predict\non D_val")
    box(11.5, 4.7, 2.2, 1.1, "AUC / F1 /\nMCC + CI")

    arrow(2.5, 5.25, 2.9, 5.25)
    arrow(5.3, 5.25, 5.7, 5.25)
    arrow(8.3, 5.25, 8.7, 5.25)
    arrow(11.1, 5.25, 11.5, 5.25)

    # Row 2 — reliability
    box(5.7, 3.0, 2.6, 1.1, "SHAP stability\nJaccard top-k", fc="#FDF0DD", ec="#B87333")
    box(8.7, 3.0, 2.4, 1.1, "SHAP–LIME\nconcordance", fc="#FDF0DD", ec="#B87333")
    box(2.9, 3.0, 2.4, 1.1, "Best-per-model\nfitted classifier", fc="#F2F2F2", ec="#666")

    arrow(4.1, 4.7, 4.1, 4.1)
    arrow(6.0, 4.1, 6.8, 4.1)
    arrow(9.0, 4.1, 9.8, 4.1)

    # Row 3 — RAMS
    box(2.9, 1.3, 3.4, 1.1, "Normalize AUC / Stab / |ρ|\nfixed ranges", fc="#E2EFDA", ec="#4C8C2B")
    box(6.9, 1.3, 3.4, 1.1, "RAMS composite\nα=0.4, β=0.3, γ=0.3", fc="#E2EFDA", ec="#4C8C2B")
    box(10.9, 1.3, 2.8, 1.1, "Rank + select\ntop model", fc="#E2EFDA", ec="#4C8C2B")

    arrow(4.1, 3.0, 4.6, 2.4)
    arrow(9.8, 3.0, 9.8, 2.4)
    arrow(6.3, 1.85, 6.9, 1.85)
    arrow(10.3, 1.85, 10.9, 1.85)

    # Row 4 — final test
    box(10.9, 0.0, 2.8, 0.9, "One-time D_test\nevaluation", fc="#FCE4D6", ec="#C00000")
    arrow(12.3, 1.3, 12.3, 0.9)

    ax.text(7, 6.3, "RAMS pipeline (three-way split, selection-unbiased)",
            ha="center", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/fig1_pipeline.png"); plt.close()
    print(f"[Fig 1] {OUT_FIG}/fig1_pipeline.png")


# =============================================================================
# FIG 2 — Validation AUC vs SHAP stability
# =============================================================================
def fig2_auc_vs_stability():
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    for ax, ds in zip(axes, DATASETS):
        d = _rank_df(ds)
        scored = d.dropna(subset=["stability_mean"])
        unscored = d[d["stability_mean"].isna()]
        if len(scored):
            ax.scatter(scored["auc"], scored["stability_mean"],
                       s=140, c="teal", edgecolors="black",
                       linewidths=1.2, label="RAMS-scored", zorder=3)
            for _, r in scored.iterrows():
                ax.annotate(r["model"], (r["auc"], r["stability_mean"]),
                            fontsize=7, ha="center", va="bottom")
        if len(unscored):
            ax.scatter(unscored["auc"], [0.5]*len(unscored),
                       s=110, c="lightgray", edgecolors="gray",
                       linewidths=1, label="n/a (excluded)", zorder=2)
            for _, r in unscored.iterrows():
                ax.annotate(r["model"], (r["auc"], 0.5),
                            fontsize=7, ha="center", va="top", color="dimgray")
        ax.set_xlabel("Validation AUC")
        ax.set_ylabel("SHAP stability (mean Jaccard)")
        ax.set_title(DS_LABEL[ds])
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(alpha=0.4)
    plt.suptitle("Accuracy vs. Explanation Stability on Validation", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/fig2_auc_vs_stability.png"); plt.close()
    print(f"[Fig 2] {OUT_FIG}/fig2_auc_vs_stability.png")


# =============================================================================
# FIG 3–5 — RAMS bar charts per dataset
# =============================================================================
def fig_rams_bars():
    fig_id = 3
    for ds in DATASETS:
        d = _rank_df(ds).dropna(subset=["RAMS"]).copy()
        if not len(d):
            # AUC-only dataset — produce bar chart of AUC
            d = _rank_df(ds).sort_values("auc", ascending=True)
            fig, ax = plt.subplots(figsize=(11, 6.5))
            colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(d)))
            ax.barh(d["model"], d["auc"], color=colors,
                    edgecolor="black", linewidth=0.6)
            for i, (_, r) in enumerate(d.iterrows()):
                ax.text(r["auc"]+0.003, i, f"{r['auc']:.3f}",
                        va="center", fontsize=8)
            ax.set_xlabel("Validation AUC")
            ax.set_title(f"{DS_LABEL[ds]} — AUC-only (low-dimensional)")
            ax.axvline(0.70, ls="--", c="red", alpha=0.5, label="AUC floor 0.70")
            ax.legend(loc="lower right")
            ax.set_xlim(0.5, d["auc"].max()*1.15)
            plt.tight_layout()
            plt.savefig(f"{OUT_FIG}/fig{fig_id}_rams_{ds.lower()}.png"); plt.close()
            print(f"[Fig {fig_id}] {OUT_FIG}/fig{fig_id}_rams_{ds.lower()}.png (AUC-only)")
            fig_id += 1
            continue

        d = d.sort_values("RAMS")
        fig, ax = plt.subplots(figsize=(11.5, 7))
        colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(d)))
        ax.barh(d["model"], d["RAMS"], color=colors,
                edgecolor="black", linewidth=0.6)
        for i, (_, r) in enumerate(d.iterrows()):
            label = (f"AUC={r['auc']:.3f}  |  Stab={r['stability_mean']:.2f}  |  "
                     f"|ρ|={r['concordance_abs_mean']:.2f}")
            ax.text(r["RAMS"]+0.005, i, label, va="center", fontsize=8)
        ax.set_xlabel("RAMS composite score (validation)")
        ax.set_title(f"{DS_LABEL[ds]} — Reliability-Aware Model Selection")
        ax.set_xlim(0, d["RAMS"].max()*1.45)
        plt.tight_layout()
        plt.savefig(f"{OUT_FIG}/fig{fig_id}_rams_{ds.lower()}.png"); plt.close()
        print(f"[Fig {fig_id}] {OUT_FIG}/fig{fig_id}_rams_{ds.lower()}.png")
        fig_id += 1


# =============================================================================
# FIG 6 — RAMS heatmap across models × datasets
# =============================================================================
def fig6_rams_heatmap():
    heat = {}
    for ds in DATASETS:
        d = _rank_df(ds).set_index("model")
        heat[ds] = d["RAMS"].fillna(d["auc"])  # use AUC for low-dim
    heat = pd.DataFrame(heat)
    # sort by Telco RAMS
    sort_key = heat["Telco"].fillna(-1)
    heat = heat.loc[sort_key.sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(7.5, 10))
    sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlGnBu",
                linewidths=0.5, cbar_kws={"label": "RAMS (or AUC for low-dim)"},
                ax=ax, mask=heat.isna())
    ax.set_title("RAMS across models and datasets\n(Superstore shown as AUC-only)")
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/fig6_rams_heatmap.png"); plt.close()
    print(f"[Fig 6] {OUT_FIG}/fig6_rams_heatmap.png")


# =============================================================================
# FIG 7 — Reliability quadrant
# =============================================================================
def fig7_quadrant():
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    for ax, ds in zip(axes, DATASETS):
        d = _rank_df(ds).dropna(subset=["stability_mean"])
        if not len(d):
            ax.text(0.5, 0.5, "Low-dimensional\n(RAMS not evaluable)",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=11, color="dimgray")
            ax.set_title(DS_LABEL[ds]); ax.set_xticks([]); ax.set_yticks([])
            continue
        ma, ms = d["auc"].median(), d["stability_mean"].median()
        ax.axvline(ma, color="gray", ls="--", lw=1)
        ax.axhline(ms, color="gray", ls="--", lw=1)
        # Shade quadrants
        ax.fill_between([ma, d["auc"].max()+0.01], ms, d["stability_mean"].max()+0.01,
                        alpha=0.06, color="green")
        ax.fill_between([ma, d["auc"].max()+0.01], 0, ms,
                        alpha=0.06, color="red")
        ax.scatter(d["auc"], d["stability_mean"], s=160, c="coral",
                   edgecolors="black", linewidths=1.2, zorder=3)
        for _, r in d.iterrows():
            ax.annotate(r["model"], (r["auc"], r["stability_mean"]),
                        fontsize=7, ha="center", va="bottom")
        ax.text(0.98, 0.98, "Accurate + Stable",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=9, style="italic", color="darkgreen")
        ax.text(0.98, 0.02, "Accurate but Unstable",
                transform=ax.transAxes, ha="right", va="bottom",
                fontsize=9, style="italic", color="darkred")
        ax.set_xlabel("Validation AUC")
        ax.set_ylabel("SHAP stability")
        ax.set_title(DS_LABEL[ds])
    plt.suptitle("Reliability quadrant on validation", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/fig7_quadrant.png"); plt.close()
    print(f"[Fig 7] {OUT_FIG}/fig7_quadrant.png")


# =============================================================================
# FIG 8 — Sensitivity
# =============================================================================
def fig8_sensitivity():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Weights
    ax = axes[0]
    for ds in ["Telco", "GermanCredit"]:
        w = R[ds]["sensitivity"]["weights"]
        alphas = [x["alpha"] for x in w]
        models = [x["top_model"] for x in w]
        y = [1 if m == models[0] else 0.5 for m in models]
        ax.plot(alphas, y, marker="o", label=DS_LABEL[ds], lw=2)
    ax.set_xlabel("α (AUC weight)")
    ax.set_ylabel("Selection stability")
    ax.set_title("Weight sensitivity (top-model stability)")
    ax.set_yticks([0.5, 1.0]); ax.set_yticklabels(["changed", "same"])
    ax.legend(); ax.grid(alpha=0.4)

    # Threshold
    ax = axes[1]
    for ds in ["Telco", "GermanCredit"]:
        t = R[ds]["sensitivity"]["threshold"]
        ax.plot([x["tau"] for x in t], [x["n_eligible"] for x in t],
                marker="o", label=DS_LABEL[ds], lw=2)
    ax.set_xlabel("AUC eligibility threshold τ")
    ax.set_ylabel("# eligible models")
    ax.set_title("Threshold sensitivity")
    ax.legend(); ax.grid(alpha=0.4)

    # Normalization
    ax = axes[2]
    labels, x_pos, colors_list = [], [], []
    for ds in ["Telco", "GermanCredit"]:
        n = R[ds]["sensitivity"]["normalization"]
        for entry in n:
            labels.append(f"{DS_LABEL[ds]}\n{entry['norm']}")
            x_pos.append(len(labels)-1)
            colors_list.append("teal" if entry["norm"] == "fixed" else "orange")
    heights = [1]*len(x_pos)
    ax.bar(x_pos, heights, color=colors_list, edgecolor="black")
    ax.set_xticks(x_pos); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_title("Normalization: fixed (teal) vs pool (orange)")
    ax.set_yticks([])

    plt.suptitle("RAMS sensitivity analysis", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/fig8_sensitivity.png"); plt.close()
    print(f"[Fig 8] {OUT_FIG}/fig8_sensitivity.png")


# =============================================================================
# FIG 9 — ΔAUC forest plot
# =============================================================================
def fig9_forest():
    fig, ax = plt.subplots(figsize=(9, 3.5))
    rows = []
    for ds in DATASETS:
        da = R[ds]["delta_auc"]
        rows.append({
            "dataset": DS_LABEL[ds],
            "mean": da["mean"], "lo": da["ci_lo"], "hi": da["ci_hi"],
        })
    df = pd.DataFrame(rows)
    y = np.arange(len(df))[::-1]
    ax.axvline(0, color="black", lw=1, ls="--")
    for i, (_, r) in enumerate(df.iterrows()):
        if np.isnan(r["lo"]) or np.isnan(r["hi"]):
            ax.scatter(r["mean"], i, s=80, c="dimgray", zorder=3)
        else:
            ax.plot([r["lo"], r["hi"]], [i, i], lw=2, color="teal", zorder=2)
            ax.scatter(r["mean"], i, s=100, c="teal",
                       edgecolors="black", zorder=3)
            ax.text(r["hi"]+0.003, i,
                    f"{r['mean']:+.4f} [{r['lo']:+.4f}, {r['hi']:+.4f}]",
                    va="center", fontsize=8)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df["dataset"].iloc[::-1])
    ax.set_xlabel("ΔAUC  (RAMS-selected − AUC-selected)")
    ax.set_title("Test-set ΔAUC with 95% bootstrap CI")
    ax.grid(alpha=0.4, axis="x")
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/fig9_forest.png"); plt.close()
    print(f"[Fig 9] {OUT_FIG}/fig9_forest.png")


# =============================================================================
# FIG 10 — Concordance distribution (signed vs |r|)
# =============================================================================
def fig10_concordance():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for ax, col, title in [
        (axes[0], "concordance_signed_mean", "Signed Spearman ρ (reported)"),
        (axes[1], "concordance_abs_mean",    "|ρ| (used in RAMS)"),
    ]:
        for ds in ["Telco", "GermanCredit"]:
            d = _rank_df(ds).dropna(subset=[col])
            if not len(d):
                continue
            ax.scatter(d["auc"], d[col], s=120, alpha=0.7,
                        edgecolors="black", linewidths=1,
                        label=DS_LABEL[ds])
        ax.axhline(0, color="black", lw=0.8, ls="--")
        ax.set_xlabel("Validation AUC")
        ax.set_ylabel(title.split("(")[0].strip())
        ax.set_title(title)
        ax.legend()
        ax.grid(alpha=0.4)
    plt.suptitle("SHAP–LIME concordance: signed vs. absolute", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/fig10_concordance.png"); plt.close()
    print(f"[Fig 10] {OUT_FIG}/fig10_concordance.png")


# =============================================================================
# TABLES
# =============================================================================
def table1_predictive():
    """Table 1 — validation AUC per model × dataset with CI."""
    rows = []
    models = sorted({m for ds in DATASETS for m in R[ds]["val_metrics"]})
    for m in models:
        row = {"Model": m}
        for ds in DATASETS:
            mtr = R[ds]["val_metrics"].get(m, {})
            if "auc_roc" in mtr:
                ci = mtr.get("auc_ci", (np.nan, np.nan))
                row[f"{DS_LABEL[ds]} AUC"] = f"{mtr['auc_roc']:.3f}"
                row[f"{DS_LABEL[ds]} 95% CI"] = f"[{ci[0]:.3f}, {ci[1]:.3f}]" \
                    if not np.isnan(ci[0]) else "—"
            else:
                row[f"{DS_LABEL[ds]} AUC"] = "—"
                row[f"{DS_LABEL[ds]} 95% CI"] = "—"
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_TAB}/table1_predictive.csv", index=False)
    print(f"[Table 1] {OUT_TAB}/table1_predictive.csv")
    return df


def table2_reliability():
    """Table 2 — stability, signed ρ, |ρ| with CIs, per model × dataset."""
    rows = []
    models = sorted({m for ds in DATASETS for m in R[ds]["reliability"]})
    for m in models:
        row = {"Model": m}
        for ds in DATASETS:
            r = R[ds]["reliability"].get(m, {})
            st  = r.get("stability_mean", np.nan)
            rho = r.get("concordance_signed_mean", np.nan)
            arb = r.get("concordance_abs_mean", np.nan)
            if not np.isnan(st):
                row[f"{DS_LABEL[ds]} Stab"]  = f"{st:.3f}"
                row[f"{DS_LABEL[ds]} Stab CI"] = (f"[{r['stability_lo']:.3f}, "
                                                  f"{r['stability_hi']:.3f}]")
            else:
                row[f"{DS_LABEL[ds]} Stab"]  = "n/a"
                row[f"{DS_LABEL[ds]} Stab CI"] = "—"
            if not np.isnan(rho):
                row[f"{DS_LABEL[ds]} ρ (signed)"] = f"{rho:+.3f}"
                row[f"{DS_LABEL[ds]} |ρ|"]        = f"{arb:.3f}"
            else:
                row[f"{DS_LABEL[ds]} ρ (signed)"] = "n/a"
                row[f"{DS_LABEL[ds]} |ρ|"]        = "n/a"
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_TAB}/table2_reliability.csv", index=False)
    print(f"[Table 2] {OUT_TAB}/table2_reliability.csv")
    return df


def table3to5_rams():
    for i, ds in enumerate(DATASETS, start=3):
        d = _rank_df(ds).copy()
        cols = ["rank", "model", "auc", "stability_mean",
                "concordance_signed_mean", "concordance_abs_mean",
                "RAMS", "Eligible"]
        present = [c for c in cols if c in d.columns]
        d_out = d[present].copy()
        d_out.to_csv(f"{OUT_TAB}/table{i}_rams_{ds.lower()}.csv", index=False)
        print(f"[Table {i}] {OUT_TAB}/table{i}_rams_{ds.lower()}.csv")


def table6_sensitivity():
    rows = []
    for ds in ["Telco", "GermanCredit"]:
        s = R[ds]["sensitivity"]
        for w in s["weights"]:
            rows.append({"Dataset": DS_LABEL[ds], "Type": "weight",
                          "Setting": f"α={w['alpha']}",
                          "Top model": w["top_model"]})
        for t in s["threshold"]:
            rows.append({"Dataset": DS_LABEL[ds], "Type": "threshold",
                          "Setting": f"τ={t['tau']}",
                          "Top model": t["top_model"]})
        for n in s["normalization"]:
            rows.append({"Dataset": DS_LABEL[ds], "Type": "normalization",
                          "Setting": n["norm"],
                          "Top model": n["top_model"]})
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_TAB}/table6_sensitivity.csv", index=False)
    print(f"[Table 6] {OUT_TAB}/table6_sensitivity.csv")
    return df


def table7_statistics():
    rows = []
    for ds in DATASETS:
        r = R[ds]
        rows.append({
            "Dataset": DS_LABEL[ds],
            "RAMS-selected": r["selected_model"],
            "AUC-selected":  r["auc_winner"],
            "Test AUC (RAMS)": f"{r['test_metrics_selected']['auc_roc']:.3f}",
            "Test AUC (AUC)":  f"{r['test_metrics_auc_winner']['auc_roc']:.3f}",
            "ΔAUC":   f"{r['delta_auc']['mean']:+.4f}",
            "ΔAUC 95% CI":
                f"[{r['delta_auc']['ci_lo']:+.4f}, {r['delta_auc']['ci_hi']:+.4f}]"
                if not np.isnan(r['delta_auc']['ci_lo']) else "—",
            "McNemar p": f"{r['mcnemar']['p_value']:.4f}",
            "n10": r["mcnemar"]["n10"], "n01": r["mcnemar"]["n01"],
        })
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_TAB}/table7_statistics.csv", index=False)
    print(f"[Table 7] {OUT_TAB}/table7_statistics.csv")
    return df


# =============================================================================
# RUN EVERYTHING
# =============================================================================
if __name__ == "__main__":
    print("="*90)
    print("GENERATING FIGURES")
    print("="*90)
    fig1_pipeline_diagram()
    fig2_auc_vs_stability()
    fig_rams_bars()
    fig6_rams_heatmap()
    fig7_quadrant()
    fig8_sensitivity()
    fig9_forest()
    fig10_concordance()

    print("\n" + "="*90)
    print("GENERATING TABLES")
    print("="*90)
    t1 = table1_predictive()
    t2 = table2_reliability()
    table3to5_rams()
    t6 = table6_sensitivity()
    t7 = table7_statistics()

    print("\n" + "="*90)
    print("[OK] All figures in ./figures/ and tables in ./tables/")
    print("="*90)

    # Print tables to console too
    print("\n--- Table 1: Predictive performance ---")
    print(t1.to_string(index=False))
    print("\n--- Table 2: Reliability ---")
    print(t2.to_string(index=False))
    print("\n--- Table 6: Sensitivity ---")
    print(t6.to_string(index=False))
    print("\n--- Table 7: Statistical comparison ---")
    print(t7.to_string(index=False))

GENERATING FIGURES
[Fig 1] figures/fig1_pipeline.png
[Fig 2] figures/fig2_auc_vs_stability.png
[Fig 3] figures/fig3_rams_telco.png
[Fig 4] figures/fig4_rams_germancredit.png
[Fig 5] figures/fig5_rams_superstore.png (AUC-only)
[Fig 6] figures/fig6_rams_heatmap.png
[Fig 7] figures/fig7_quadrant.png
[Fig 8] figures/fig8_sensitivity.png
[Fig 9] figures/fig9_forest.png
[Fig 10] figures/fig10_concordance.png

GENERATING TABLES
[Table 1] tables/table1_predictive.csv
[Table 2] tables/table2_reliability.csv
[Table 3] tables/table3_rams_telco.csv
[Table 4] tables/table4_rams_germancredit.csv
[Table 5] tables/table5_rams_superstore.csv
[Table 6] tables/table6_sensitivity.csv
[Table 7] tables/table7_statistics.csv

[OK] All figures in ./figures/ and tables in ./tables/

--- Table 1: Predictive performance ---
             Model Telco AUC   Telco 95% CI German Credit AUC German Credit 95% CI Superstore AUC Superstore 95% CI
          CatBoost     0.839 [0.814, 0.860]             0.837       [0.777,